# Tool-trajectory evaluation before agent release

This credential-free lab adapts MLflow's [LangGraph agent cookbook](https://mlflow.org/cookbook/langgraph-agent/) to the platform contract. A fluent final answer is not enough: an agent can quote the right fact while calling an unapproved tool, using the wrong arguments, or skipping a required lookup.

The lab starts with deterministic exact matching because release gates need reproducible evidence. A connected project may add MLflow's fuzzy `ToolCallCorrectness` judge as report-only evidence after routing it through the governed judge model.

The assurance model keeps **Outcome** (the final answer), **Behavior** (decisions and executed tools), **Operations** (latency, tokens, cost, failures, and policy), and **Optional internal diagnostics** (provider-supported debugging signals) distinct. An independent **Assessment** determines whether the recorded outcome and behavior were acceptable. Provider diagnostics are never required, and this lab does not request, reconstruct, infer, or persist hidden chain-of-thought.

The repository's executable LangGraph implementation remains the optional native recipe under `templates/agent-app/template/recipes/langgraph/`. It owns graph state, durable checkpoints, interrupts, and idempotency; `aai-core` does not wrap those APIs.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples" / "support").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open the cloned repository as your workspace.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Define outcome, decision, and execution evidence

Each row keeps `expected_facts` and expected tools separate from the application's concise decision records and the observed tool calls. A decision says what the application chose and why; it does not say that the chosen tool succeeded. Observed execution remains authoritative.

The second case is intentional: its answer contains the right number and its declared choice matches the tool that ran, but the cached-summary tool was inappropriate for the benchmark. Output-only evaluation and action-consistency evaluation would both miss that failure. The third case makes the correct tool decision, observes that TOOL execution failed, and records an application-authored safe fallback. It does not imply that an SDK or provider recovers by default. Each row has partial Operations evidence because TOOL status is observed, while latency, token, and cost remain `None`, never zero. An otherwise identical row with no observed status or telemetry would remain `unknown`.

In [ ]:
from examples.support.agent_assurance import tool_trajectory_cases

EVAL_CASES = tool_trajectory_cases()
assert [case["case_id"] for case in EVAL_CASES] == [
    "revenue-from-governed-source",
    "right-answer-wrong-trajectory",
    "correct-tool-failed-safe-fallback",
]

## 2. Assess decision claims separately from observed execution

`decision_action_consistency` asks whether the tool selected in the decision record matches the tool that actually ran. It does not grade whether the selection was good. `decision_tool_appropriateness` compares the selected tool with the benchmark expectation, while `tool_trajectory_exact` also checks arguments. `tool_execution_succeeded` comes only from the observed result. `safe_fallback_observed` requires matching fallback/readiness records and an ordered fixture trajectory in which both decisions occur after the matching errored TOOL result.

The assessments below are calculated independently; they do not mutate the original decision record. A multiset preserves duplicate calls, so calling the same expensive tool twice cannot collapse into one passing set member. Ordering is ignored here because these lookups are independent; make ordering explicit when tool sequence is part of the contract. An explicit `ok` or `error` TOOL status is partial Operations evidence when latency, token, and cost telemetry is absent; no status and no telemetry remains `unknown`. This is an offline decision fixture, not a fabricated MLflow trace.

In [ ]:
from examples.support.agent_assurance import (
    build_tool_trajectory_reports,
    score_tool_trajectory_case,
)

score_case = score_tool_trajectory_case
trajectory_report, assurance_report = build_tool_trajectory_reports(EVAL_CASES)
assurance_report

In [ ]:
from examples.support.agent_assurance import tool_trajectory_gate

report_by_case = trajectory_report.set_index("case_id")
wrong_path = report_by_case.loc["right-answer-wrong-trajectory"]
safe_fallback = report_by_case.loc["correct-tool-failed-safe-fallback"]
assert trajectory_report["final_answer_correct"].all()
assert trajectory_report["decision_action_consistency"].all()
assert not bool(wrong_path["decision_tool_appropriateness"])
assert not bool(wrong_path["tool_trajectory_exact"])
assert not bool(safe_fallback["tool_execution_succeeded"])
assert bool(safe_fallback["safe_fallback_observed"])
assert safe_fallback["operations_assessment"] == "FAIL"

trajectory_gate_result = tool_trajectory_gate(trajectory_report)
gate_passed = trajectory_gate_result["gate_passed"]
lifecycle_decision = trajectory_gate_result["decision"]
trajectory_gate_result

## 3. Carry the contract into the connected agent template

In a generated `agent-app`, use `app.tool_scoring.exact_tool_call_scorer()`, `decision_action_consistency_scorer()`, and `decision_tool_appropriateness_scorer()` in the release gate. The native MLflow evaluation rows use the same `inputs`, `expected_facts`, and `expected_tool_calls` shape shown above; the decision-appropriateness scorer reuses the existing expected tools instead of adding duplicate ground truth.

For the optional LangGraph recipe:

1. Install its certified lock and inject an async durable checkpointer and persistent store.
2. Configure only `TraceIntegration.MLFLOW_LANGCHAIN` with `run_tracer_inline=True`; do not add SDK provider spans around the same graph call.
3. Give every `ainvoke()` or resume its own trace context while reusing the opaque checkpoint/session ID.
4. Keep interrupts before side effects and protect resumed execution with an idempotency key.
5. Evaluate traced calls with the deterministic exact scorer. If fuzzy `ToolCallCorrectness` is also useful, route it through the configured judge model and keep it report-only until calibrated.

This lab rejects the observed change because one decision selected an inappropriate tool and one critical trajectory failed, even though final-answer correctness and decision/action consistency are both 100%. The per-request `tool_selection`, after-tool `evidence_sufficiency`, failure `fallback`, and terminal `answer_readiness` records are runtime agent decisions. The third fixture explicitly orders an application-authored safe fallback and answer-readiness decision after the matching observed error; it does not claim that MLflow, Agent Framework, Foundry, or the model provides default recovery. A connected trace must show the failed TOOL execution and subsequent application decisions in that order before a reviewer claims recovery. The `reject` value produced by the gate is a later lifecycle release decision based on independent assessments.

The optional cell below persists this synthetic contract as a Unity Catalog EvaluationDataset and records a described MLflow result run. It creates no agent calls, prompts, or traces.

In [ ]:
from examples.support.agent_assurance import persist_tool_trajectory_evidence

PERSIST_EVIDENCE_TO_DATABRICKS = False
if PERSIST_EVIDENCE_TO_DATABRICKS:
    print(persist_tool_trajectory_evidence(EVAL_CASES, trajectory_report))
else:
    print("DATABRICKS EVIDENCE PERSISTENCE SKIPPED")